# 예제 franka_ex11: FR3 MoveIt Servo 텔레오퍼레이션 (notebook 버전)

원본 6-DOF `ex11_keyboard_servo.py` 는 curses 기반 인터랙티브 키보드 텔레오퍼레이션이라
주피터 노트북에 그대로 옮길 수 없다. 대신 **셀에서 명시적으로 명령을 발행하는 프로그램형 텔레오퍼레이션**
형태로 옮긴다 — 본질(TwistStamped → MoveIt Servo → 컨트롤러) 은 같다.

**사전 준비 — 별도 Servo 노드 필요**

`franka_tutorials` 의 기본 launch 에는 `moveit_servo` 가 포함되어 있지 않다.
이 노트북을 동작시키려면 **터미널 1 외에 추가로** Servo 노드를 띄워야 한다.
빠르게 띄우려면 다음 ros2 명령을 참고하자 (직접 실행이 막히면 launch 파일을 새로 작성).

```bash
ros2 run moveit_servo servo_node \
  --ros-args -p use_sim_time:=true \
  -p moveit_servo.move_group_name:=fr3_arm \
  -p moveit_servo.planning_frame:=fr3_link0 \
  -p moveit_servo.ee_frame_name:=fr3_hand_tcp \
  -p moveit_servo.command_in_type:=speed_units \
  -p moveit_servo.command_out_topic:=/fr3_arm_controller/joint_trajectory \
  -p moveit_servo.command_out_type:=trajectory_msgs/JointTrajectory \
  -p moveit_servo.publish_joint_positions:=true \
  -p moveit_servo.publish_joint_velocities:=false
```

> Servo 가 띄워지지 않은 상태에서 이 노트북을 돌리면, TwistStamped 메시지는 발행되지만 로봇은 움직이지 않는다.
> Servo 가 부담스럽다면 `franka_ex15` 의 IK 직접 스트리밍 방식이 더 가볍다.

**6-DOF 예제와 다른 점**
- planning_frame `fr3_link0`, ee_frame `fr3_hand_tcp`, group `fr3_arm`
- 7-DOF redundancy 덕에 같은 명령에서 특이점 회피 여유가 더 크다
- ready 자세는 SRDF 의 `ready` 사용 (6-DOF 와 달리 `home` 없음)
- TwistStamped 발행 시 `frame_id='fr3_link0'`

**학습 내용**
- `TwistStamped` 메시지 구조 (`twist.linear`, `twist.angular`)
- `/servo_node/delta_twist_cmds` 토픽의 의미
- 30Hz 주기로 Twist 명령을 일정 시간 발행 → 끝단을 그 방향으로 슬슬 미는 효과
- RViz `MarkerArray` 로 끝단 누적 경로 시각화 (TF + 직접 lookup)

## 실행 절차

이 노트북은 별도로 띄운 MoveIt + RViz 의 `move_group` 액션 서버에 클라이언트로 붙는다.

> ⚠ 다른 로봇용 MoveIt launch 가 떠 있으면 같은 토픽으로 충돌할 수 있다.
> 시작 전에 `pgrep -af 'ros2 launch'` 로 잔존 프로세스가 없는지 확인하자.

### 터미널 1 — Franka FR3 (Gazebo Sim) + MoveIt + RViz 기동

```bash
source /opt/ros/jazzy/setup.bash
source ~/robot_arm/install/setup.bash
ros2 launch franka_tutorials franka_gazebo_moveit.launch.py
```

RViz 가 뜨면 **`MarkerArray` Display 를 추가하고 Topic 을 `/servo_viz_markers` 로 설정**한다.
Fixed Frame 은 `fr3_link0` 로 둔다.

TF Display 도 켜 두면 끝단이 어디 있는지 한눈에 보기 좋다.

### 터미널 2 — Jupyter 기동

```bash
source ~/venv/ros_jazzy/bin/activate
source /opt/ros/jazzy/setup.bash
source ~/robot_arm/install/setup.bash
cd ~/robot_arm/src/robotarm_tutorials/robot_arm_tutorials/robot_arm_tutorials
jupyter lab
```

셀을 위에서 아래로 순서대로 실행한다 (`Shift+Enter`).

## 1. 상수

In [ ]:
PLANNING_GROUP    = 'fr3_arm'
REFERENCE_FRAME   = 'fr3_link0'
END_EFFECTOR_LINK = 'fr3_hand_tcp'
ARM_JOINTS        = ['fr3_joint1', 'fr3_joint2', 'fr3_joint3',
                     'fr3_joint4', 'fr3_joint5', 'fr3_joint6',
                     'fr3_joint7']
SERVO_TWIST_TOPIC = '/servo_node/delta_twist_cmds'
SWITCH_CMD_SRV    = '/servo_node/switch_command_type'
MARKER_TOPIC      = '/servo_viz_markers'
COMMAND_TYPE_TWIST = 1   # moveit_servo CommandType.TWIST

## 2. 초기화

In [ ]:
import rclpy
import math, time, threading
from rclpy.node import Node
from rclpy.action import ActionClient
from rclpy.parameter import Parameter
from sensor_msgs.msg import JointState
from geometry_msgs.msg import TwistStamped, Point, Vector3
from std_msgs.msg import ColorRGBA
from visualization_msgs.msg import Marker, MarkerArray
from moveit_msgs.action import MoveGroup
from moveit_msgs.srv import ServoCommandType
import tf2_ros

In [ ]:
try:
    rclpy.init()
except RuntimeError:
    pass  # 이미 초기화된 경우 무시

In [ ]:
node = Node(
    'franka_ex11_servo_demo',
    parameter_overrides=[Parameter('use_sim_time', value=True)],
)
move_client = ActionClient(node, MoveGroup, 'move_action')

joint_state = {'msg': None}
node.create_subscription(
    JointState, 'joint_states',
    lambda msg: joint_state.update(msg=msg), 10,
)
node.get_logger().info('=== franka_ex11 노트북 노드 생성 완료 ===')
marker_pub = node.create_publisher(MarkerArray, MARKER_TOPIC, 10)
twist_pub = node.create_publisher(TwistStamped, SERVO_TWIST_TOPIC, 10)
switch_cmd = node.create_client(ServoCommandType, SWITCH_CMD_SRV)
tf_buffer = tf2_ros.Buffer()
tf_listener = tf2_ros.TransformListener(tf_buffer, node)

## 3. 서버 / `joint_states` 준비

In [ ]:
import time

def wait_for_ready(timeout_sec: float = 30.0) -> None:
    if not move_client.wait_for_server(timeout_sec=timeout_sec):
        raise RuntimeError('MoveGroup 액션 서버 연결 실패')
    start = time.time()
    while joint_state['msg'] is None:
        rclpy.spin_once(node, timeout_sec=0.1)
        if time.time() - start > timeout_sec:
            raise RuntimeError('joint_states 수신 실패')
    node.get_logger().info('action server + /joint_states 준비됨')

wait_for_ready()

## 4. SRDF `ready` 자세

In [ ]:
from rclpy.parameter_client import AsyncParameterClient
import xml.etree.ElementTree as ET

def fetch_srdf_xml(timeout_sec: float = 10.0) -> str:
    client = AsyncParameterClient(node, 'move_group')
    if not client.wait_for_services(timeout_sec=timeout_sec):
        raise RuntimeError('move_group 파라미터 서비스 연결 실패')
    future = client.get_parameters(['robot_description_semantic'])
    rclpy.spin_until_future_complete(node, future, timeout_sec=timeout_sec)
    return future.result().values[0].string_value

def parse_named_pose(srdf_xml: str, name: str, group: str) -> dict:
    root = ET.fromstring(srdf_xml)
    for gs in root.findall('group_state'):
        if gs.attrib.get('group') == group and gs.attrib.get('name') == name:
            return {j.attrib['name']: float(j.attrib.get('value', '0'))
                    for j in gs.findall('joint')}
    raise RuntimeError(f'SRDF group_state "{name}" (group={group}) 없음')

def load_named_pose(name: str, timeout_sec: float = 10.0) -> dict:
    return parse_named_pose(fetch_srdf_xml(timeout_sec), name, PLANNING_GROUP)

ready_target = load_named_pose('ready')
node.get_logger().info(f'ready: {ready_target}')

## 5. Pose / MoveGroup 헬퍼

In [ ]:
import math
import tf_transformations
from geometry_msgs.msg import Pose, Point, Quaternion

def euler_to_quaternion(roll: float, pitch: float, yaw: float) -> Quaternion:
    q = tf_transformations.quaternion_from_euler(roll, pitch, yaw)
    return Quaternion(x=q[0], y=q[1], z=q[2], w=q[3])

def make_pose(x: float, y: float, z: float,
              roll: float = 0.0, pitch: float = 0.0, yaw: float = 0.0) -> Pose:
    pose = Pose()
    pose.position = Point(x=x, y=y, z=z)
    pose.orientation = euler_to_quaternion(roll, pitch, yaw)
    return pose

In [ ]:
from moveit_msgs.msg import (
    Constraints, JointConstraint,
    PositionConstraint, OrientationConstraint, BoundingVolume,
    MotionPlanRequest, PlanningOptions, MoveItErrorCodes,
)
from shape_msgs.msg import SolidPrimitive
from geometry_msgs.msg import Vector3

def make_joint_constraints(joint_values: dict, tol: float = 0.01) -> Constraints:
    c = Constraints()
    for jname, val in joint_values.items():
        c.joint_constraints.append(JointConstraint(
            joint_name=jname, position=val,
            tolerance_above=tol, tolerance_below=tol, weight=1.0,
        ))
    return c

def make_position_constraint(pose: Pose, tol: float = 0.01) -> PositionConstraint:
    pc = PositionConstraint()
    pc.header.frame_id = REFERENCE_FRAME
    pc.link_name = END_EFFECTOR_LINK
    pc.target_point_offset = Vector3(x=0.0, y=0.0, z=0.0)
    bv = BoundingVolume()
    sphere = SolidPrimitive()
    sphere.type = SolidPrimitive.SPHERE
    sphere.dimensions = [tol]
    bv.primitives.append(sphere)
    sp = Pose()
    sp.position = Point(x=pose.position.x, y=pose.position.y, z=pose.position.z)
    sp.orientation.w = 1.0
    bv.primitive_poses.append(sp)
    pc.constraint_region = bv
    pc.weight = 1.0
    return pc

def make_orientation_constraint(pose_or_quat, tol: float = 0.01) -> OrientationConstraint:
    oc = OrientationConstraint()
    oc.header.frame_id = REFERENCE_FRAME
    oc.link_name = END_EFFECTOR_LINK
    if hasattr(pose_or_quat, 'orientation'):
        oc.orientation = pose_or_quat.orientation
    else:
        oc.orientation = pose_or_quat
    oc.absolute_x_axis_tolerance = tol
    oc.absolute_y_axis_tolerance = tol
    oc.absolute_z_axis_tolerance = tol
    oc.weight = 1.0
    return oc

def make_plan_request(vel: float = 0.3, acc: float = 0.3,
                      attempts: int = 5, plan_time: float = 10.0,
                      planner_id: str = '') -> MotionPlanRequest:
    req = MotionPlanRequest()
    req.group_name = PLANNING_GROUP
    req.num_planning_attempts = attempts
    req.allowed_planning_time = plan_time
    req.max_velocity_scaling_factor = vel
    req.max_acceleration_scaling_factor = acc
    if planner_id:
        req.planner_id = planner_id
    return req

def send_move_goal(req: MotionPlanRequest, plan_only: bool = False):
    goal = MoveGroup.Goal()
    goal.request = req
    goal.planning_options = PlanningOptions(
        plan_only=plan_only, replan=not plan_only, replan_attempts=3 if not plan_only else 0)
    sf = move_client.send_goal_async(goal)
    rclpy.spin_until_future_complete(node, sf)
    handle = sf.result()
    if handle is None or not handle.accepted:
        return MoveItErrorCodes.PLANNING_FAILED, None
    rf = handle.get_result_async()
    rclpy.spin_until_future_complete(node, rf)
    res = rf.result().result
    return res.error_code.val, res.planned_trajectory

def go_to_joint_goal(joint_values: dict, vel: float = 0.3, acc: float = 0.3) -> bool:
    req = make_plan_request(vel, acc)
    req.goal_constraints.append(make_joint_constraints(joint_values))
    code_val, _ = send_move_goal(req, plan_only=False)
    ok = (code_val == MoveItErrorCodes.SUCCESS)
    if not ok:
        node.get_logger().error(f'joint goal 실패 error_code={code_val}')
    return ok

def go_to_pose_goal(pose: Pose, vel: float = 0.3, acc: float = 0.3) -> bool:
    req = make_plan_request(vel, acc)
    c = Constraints()
    c.position_constraints.append(make_position_constraint(pose))
    c.orientation_constraints.append(make_orientation_constraint(pose))
    req.goal_constraints.append(c)
    code_val, _ = send_move_goal(req, plan_only=False)
    ok = (code_val == MoveItErrorCodes.SUCCESS)
    if not ok:
        node.get_logger().error(f'pose goal 실패 error_code={code_val} (IK 해 없음 가능)')
    return ok

def plan_to_joint_goal(joint_values: dict, vel: float = 0.3, acc: float = 0.3,
                       planner_id: str = '', plan_time: float = 10.0):
    req = make_plan_request(vel, acc, plan_time=plan_time, planner_id=planner_id)
    req.goal_constraints.append(make_joint_constraints(joint_values))
    code_val, traj = send_move_goal(req, plan_only=True)
    return code_val == MoveItErrorCodes.SUCCESS, traj

def plan_to_pose_goal(pose: Pose, vel: float = 0.3, acc: float = 0.3,
                      planner_id: str = '', plan_time: float = 10.0):
    req = make_plan_request(vel, acc, plan_time=plan_time, planner_id=planner_id)
    c = Constraints()
    c.position_constraints.append(make_position_constraint(pose))
    c.orientation_constraints.append(make_orientation_constraint(pose))
    req.goal_constraints.append(c)
    code_val, traj = send_move_goal(req, plan_only=True)
    return code_val == MoveItErrorCodes.SUCCESS, traj

## 6. ready 자세로 이동 (Servo 시작 전)

Servo 는 시작 자세가 특이점 근처면 즉시 정지하므로, MoveGroup 으로 안전한 자세까지 옮겨두고 시작한다.
SRDF `ready` 가 무난하다 (FR3 의 표준 시작 자세).

In [ ]:
go_to_joint_goal(ready_target, vel=0.3)
time.sleep(1.0)

## 7. Servo 명령 모드 → TWIST 전환

Servo 노드에 `ServoCommandType.TWIST` 로 모드 전환을 요청한다.
Servo 노드가 띄워져 있지 않으면 여기서 timeout 으로 실패하므로,
`Servo 노드를 먼저 띄웠는지` 다시 한번 확인하자.

In [ ]:
def switch_to_twist_mode() -> bool:
    if not switch_cmd.wait_for_service(timeout_sec=10.0):
        node.get_logger().error('switch_command_type 서비스 연결 실패 — Servo 노드가 떠 있는지 확인')
        return False
    req = ServoCommandType.Request()
    req.command_type = COMMAND_TYPE_TWIST
    fut = switch_cmd.call_async(req)
    rclpy.spin_until_future_complete(node, fut, timeout_sec=5.0)
    if fut.result() is not None and fut.result().success:
        node.get_logger().info('Servo TWIST 모드 전환 성공')
        return True
    node.get_logger().error('Servo TWIST 모드 전환 실패')
    return False

servo_ready = switch_to_twist_mode()

## 8. Twist 발행 헬퍼

`drive(linear=(vx, vy, vz), angular=(wx, wy, wz), duration_sec=2.0)` 한 번 호출하면
30Hz 로 `duration_sec` 동안 같은 Twist 를 계속 발행한다 (Servo 가 키 input 처럼 반응).
끝나면 0 Twist 를 한 번 보내 정지시킨다.

In [ ]:
TWIST_RATE_HZ = 30.0

def drive(linear=(0.0, 0.0, 0.0), angular=(0.0, 0.0, 0.0),
          duration_sec: float = 2.0):
    if not servo_ready:
        node.get_logger().error('Servo 미준비 상태에서 호출됨 (Servo 노드가 떠 있는지 확인)')
        return
    period = 1.0 / TWIST_RATE_HZ
    end = time.time() + duration_sec
    msg = TwistStamped()
    msg.header.frame_id = REFERENCE_FRAME
    msg.twist.linear.x, msg.twist.linear.y, msg.twist.linear.z = linear
    msg.twist.angular.x, msg.twist.angular.y, msg.twist.angular.z = angular
    while time.time() < end:
        msg.header.stamp = node.get_clock().now().to_msg()
        twist_pub.publish(msg)
        rclpy.spin_once(node, timeout_sec=period * 0.5)
        time.sleep(max(0.0, period - period * 0.5))
    # 정지
    stop = TwistStamped()
    stop.header.frame_id = REFERENCE_FRAME
    stop.header.stamp = node.get_clock().now().to_msg()
    twist_pub.publish(stop)

## 9. RViz 마커 — 끝단 명령 화살표 + 누적 경로

In [ ]:
COLOR_LINEAR  = ColorRGBA(r=0.0, g=1.0, b=1.0, a=1.0)   # cyan
COLOR_ANGULAR = ColorRGBA(r=1.0, g=0.0, b=1.0, a=1.0)   # magenta
COLOR_PATH    = ColorRGBA(r=0.2, g=1.0, b=0.4, a=0.85)
LINEAR_LEN    = 0.20
ANG_LEN       = 0.15

ee_path = []
def lookup_ee():
    try:
        tr = tf_buffer.lookup_transform(REFERENCE_FRAME, END_EFFECTOR_LINK, rclpy.time.Time())
    except Exception:
        return None
    t = tr.transform.translation
    return (t.x, t.y, t.z)

def publish_command_markers(linear=(0.0, 0.0, 0.0), angular=(0.0, 0.0, 0.0)):
    ma = MarkerArray()
    stamp = node.get_clock().now().to_msg()
    origin = lookup_ee()
    if origin is None:
        return
    # 누적 경로 LINE_STRIP
    if ee_path and ee_path[-1] != origin:
        ee_path.append(origin)
    elif not ee_path:
        ee_path.append(origin)
    if len(ee_path) > 2000:
        del ee_path[:len(ee_path) - 2000]

    line = Marker()
    line.header.frame_id = REFERENCE_FRAME
    line.header.stamp = stamp
    line.ns = 'ee_path'
    line.id = 0
    line.type = Marker.LINE_STRIP
    line.action = Marker.ADD
    line.pose.orientation.w = 1.0
    line.scale.x = 0.005
    line.color = COLOR_PATH
    line.points = [Point(x=p[0], y=p[1], z=p[2]) for p in ee_path]
    ma.markers.append(line)

    if abs(linear[0]) + abs(linear[1]) + abs(linear[2]) > 0.01:
        arrow = Marker()
        arrow.header.frame_id = REFERENCE_FRAME
        arrow.header.stamp = stamp
        arrow.ns = 'cmd_linear'
        arrow.id = 0
        arrow.type = Marker.ARROW
        arrow.action = Marker.ADD
        arrow.pose.orientation.w = 1.0
        arrow.points = [
            Point(x=origin[0], y=origin[1], z=origin[2]),
            Point(x=origin[0] + linear[0] * LINEAR_LEN,
                  y=origin[1] + linear[1] * LINEAR_LEN,
                  z=origin[2] + linear[2] * LINEAR_LEN),
        ]
        arrow.scale = Vector3(x=0.012, y=0.025, z=0.030)
        arrow.color = COLOR_LINEAR
        ma.markers.append(arrow)
    if abs(angular[0]) + abs(angular[1]) + abs(angular[2]) > 0.01:
        ax = Marker()
        ax.header.frame_id = REFERENCE_FRAME
        ax.header.stamp = stamp
        ax.ns = 'cmd_angular'
        ax.id = 0
        ax.type = Marker.ARROW
        ax.action = Marker.ADD
        ax.pose.orientation.w = 1.0
        h = ANG_LEN * 0.5
        ax.points = [
            Point(x=origin[0] - angular[0] * h, y=origin[1] - angular[1] * h, z=origin[2] - angular[2] * h),
            Point(x=origin[0] + angular[0] * h, y=origin[1] + angular[1] * h, z=origin[2] + angular[2] * h),
        ]
        ax.scale = Vector3(x=0.008, y=0.020, z=0.020)
        ax.color = COLOR_ANGULAR
        ma.markers.append(ax)
    marker_pub.publish(ma)

## 10. 시연 — 끝단을 슬슬 밀어 보기

각 셀은 약 2초씩 같은 방향으로 Servo 명령을 보낸다.
실행할 때마다 RViz 의 `/servo_viz_markers` 에서 끝단 위치/명령 화살표가 갱신된다.

### 10-1. 전진 (`+X`) 2초

In [ ]:
publish_command_markers(linear=(1.0, 0.0, 0.0))
drive(linear=(0.5, 0.0, 0.0), duration_sec=2.0)
publish_command_markers()  # 화살표 제거

### 10-2. 좌측 (`+Y`) 2초

In [ ]:
publish_command_markers(linear=(0.0, 1.0, 0.0))
drive(linear=(0.0, 0.5, 0.0), duration_sec=2.0)
publish_command_markers()

### 10-3. 상승 (`+Z`) 2초

In [ ]:
publish_command_markers(linear=(0.0, 0.0, 1.0))
drive(linear=(0.0, 0.0, 0.4), duration_sec=2.0)
publish_command_markers()

### 10-4. 끝단 회전 — yaw `+Z` 2초

In [ ]:
publish_command_markers(angular=(0.0, 0.0, 1.0))
drive(angular=(0.0, 0.0, 0.5), duration_sec=2.0)
publish_command_markers()

## 11. 자유 명령 — 셀에 직접 값 넣어 시도

`linear`/`angular` 에 -1.0 ~ 1.0 사이 단위 벡터를 주고 `duration_sec` 으로 길이 조절.

In [ ]:
# 예: 우측 + 하강 동시에 1.5초
publish_command_markers(linear=(0.0, -0.7, -0.7))
drive(linear=(0.0, -0.3, -0.3), duration_sec=1.5)
publish_command_markers()

## 12. ready 복귀 (MoveGroup 으로)

In [ ]:
# Servo 가 명령을 잡고 있을 수 있으므로 0 Twist 를 한 번 더 보내고,
# MoveGroup 으로 ready 자세로 복귀한다.
stop = TwistStamped()
stop.header.frame_id = REFERENCE_FRAME
stop.header.stamp = node.get_clock().now().to_msg()
twist_pub.publish(stop)
time.sleep(0.2)

go_to_joint_goal(ready_target, vel=0.3)
node.get_logger().info('=== franka_ex11 완료! ===')

## 13. 정리

In [ ]:
node.destroy_node()
try:
    rclpy.shutdown()
except Exception:
    pass